# Judge validation — MANUAL STEP (do this once, before quoting any judge numbers)

Optional. Generates a blind labelling sheet, you fill in your own behaviour/coherence scores
without looking at the judge's, then part 2 reports agreement (Spearman ρ, mean|diff|). Requires
`ANTHROPIC_API_KEY` and `scipy` (`uv pip install scipy`). Run **after** both behaviour notebooks so
`dec_dirs` / `hs_dirs` exist — or adapt to load from the exported bundles.

This is the original notebook's §"Judge validation" step, kept as a standalone optional notebook.

In [ ]:
# --- make the safety_probes package importable ---
import sys, pathlib
try:
    import safety_probes
except ModuleNotFoundError:
    starts = []
    _f = globals().get("__vsc_ipynb_file__")
    if _f: starts.append(pathlib.Path(_f).resolve())
    starts.append(pathlib.Path.cwd())
    for _s in starts:
        _root = next((p for p in [_s, *_s.parents] if (p / "safety_probes" / "__init__.py").exists()), None)
        if _root:
            sys.path.insert(0, str(_root)); break
    import safety_probes
print("safety_probes:", safety_probes.__file__)

In [ ]:
# Requires dec_dirs, dec_id_prompts, hs_dirs, hs_id_prompts, DECEPTION_SPEC, HIGH_STAKES_SPEC in scope.
# Run the deception and high_stakes notebooks first (same kernel), or reconstruct these from the
# exported bundles. Then:
import numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm
from safety_probes import runtime
from safety_probes.steering import SteeringHook, generate_steered
from safety_probes.judge import judge_generation, prompt_for_judge, get_client

JV_BLIND = runtime.PROJECT_ROOT / "judge_validation_blind.csv"
JV_JUDGE = runtime.PROJECT_ROOT / "judge_validation_judge_scores.csv"
# Judge validation part 1: create the blind labelling sheet (idempotent; skips without a judge).
if get_client() is None:
    print("No ANTHROPIC_API_KEY - judge validation skipped.")
elif JV_BLIND.exists():
    print(f"{JV_BLIND.name} already exists - not overwriting (delete both CSVs to regenerate).")
else:
    combos = [(spec, dirs, method, coef, p)
              for spec, dirs, prompts in [(DECEPTION_SPEC, dec_dirs, dec_id_prompts),
                                          (HIGH_STAKES_SPEC, hs_dirs, hs_id_prompts)]
              for method in ("MM", "LR") for coef in (0.0, 1.0, 2.0) for p in prompts[:2]]
    rows, judge_rows = [], []
    for k, (spec, dirs, method, coef, prompt) in enumerate(tqdm(combos, desc="judge-validation generations")):
        hook = None if coef == 0 else SteeringHook(dirs[method], coef=coef)
        gen = generate_steered(prompt, hook)
        j = judge_generation(spec, prompt, gen)
        rows.append({"id": k, "behaviour": spec.name, "prompt": prompt_for_judge(prompt)[-400:], "generation": gen,
                     "your_behaviour": "", "your_coherence": ""})
        judge_rows.append({"id": k, "method": method, "coef": coef,
                           "judge_behaviour": j["behaviour"] if j else np.nan,
                           "judge_coherence": j["coherence"] if j else np.nan})
    pd.DataFrame(rows).to_csv(JV_BLIND, index=False)
    pd.DataFrame(judge_rows).to_csv(JV_JUDGE, index=False)
    print(f"Wrote {len(rows)} rows.\nNOW: open {JV_BLIND.name}, fill your_behaviour/your_coherence (1-10)"
          f" WITHOUT looking at {JV_JUDGE.name}, save, then run the next cell.")

In [ ]:
# Judge validation part 2: agreement between your blind labels and the judge's scores.
import numpy as np, pandas as pd
if not (JV_BLIND.exists() and JV_JUDGE.exists()):
    print("Run the cell above first.")
else:
    merged = pd.read_csv(JV_BLIND).merge(pd.read_csv(JV_JUDGE), on="id")
    merged = merged.dropna(subset=["your_behaviour", "your_coherence", "judge_behaviour", "judge_coherence"])
    if len(merged) == 0:
        print("No filled rows yet - label judge_validation_blind.csv first (see instructions above).")
    else:
        from scipy.stats import spearmanr
        for axis in ("behaviour", "coherence"):
            a = merged[f"your_{axis}"].astype(float); b = merged[f"judge_{axis}"].astype(float)
            print(f"{axis}: n={len(merged)}  Spearman rho={spearmanr(a, b).statistic:.2f}  "
                  f"mean|diff|={np.abs(a - b).mean():.2f}  within +-2: {(np.abs(a - b) <= 2).mean():.0%}")